# DSPy Embeddings — One Interface, Three Ways

**Week 6 | Notebook 8 of 12**

**What you'll learn:**
- What `dspy.Embedder` is and why DSPy unifies embedding backends behind one class
- Hosted models via litellm (OpenAI `text-embedding-3-small`)
- Local, offline models via sentence-transformers
- Plugging in any custom embedding function
- Batching and caching behavior
- Reading embeddings: cosine similarity between phrases

**Runtime:** ~10 minutes

**Note:** Based on the official API reference:
[dspy.Embedder](https://dspy.ai/current/api/models/Embedder/). The hosted example makes
~3 tiny API calls (fractions of a cent); examples 2 and 3 run entirely on your machine.

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/08_embeddings.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/08_embeddings.ipynb
Task:      Embeddings: hosted + local + custom
Calls:     ~4

With GPT-4o:       $0.02 USD
With GPT-4o-mini:  $0.00 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [2]:
import dspy
import numpy as np

from src.config import get_embedder, print_config

print_config()

# Our unified embedder — defaults to openai/text-embedding-3-small,
# override with EMBEDDING_MODEL in .env (litellm routes any supported model)
embedder = get_embedder()

print(f"\n✅ Embedder configured with: {embedder.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      openai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.6-flash
  groq model:      openai/gpt-oss-120b
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ Embedder configured with: openai/text-embedding-3-small


## 2. What Is `dspy.Embedder`?

An embedding turns a piece of text into a vector of floats such that *semantically
similar* texts land *geometrically close* together. `dspy.Embedder` is DSPy's unified
interface for computing them. It accepts **two kinds of model**:

1. **Hosted models** — pass a litellm model name (e.g. `"openai/text-embedding-3-small"`).
   litellm handles the API call and caching, using the provider key from your `.env`.
2. **Custom callables** — pass any function that takes a list of strings and returns a 2D
   array (or list) of float32 rows. Local models, your own service, anything.

Constructor:

| Param | Type | Default | Meaning |
|---|---|---|---|
| `model` | `str \| Callable` | *required* | litellm model name, or a callable `f(texts) -> 2D floats` |
| `batch_size` | `int` | `200` | inputs processed per batch for hosted models |
| `caching` | `bool` | `True` | cache hosted responses (re-runs are free) |
| `**kwargs` | `dict` | `{}` | default arguments forwarded to the model on every call |

Calling contract — `__call__(inputs, batch_size=None, caching=None, **kwargs)`:

- a **single string** returns a 1D `np.ndarray` (one vector)
- a **list of strings** returns a 2D `np.ndarray`, one row per input
- `acall(...)` is the async variant for high-throughput pipelines

The three examples below are the official ones from the docs page.

## 3. Official Example 1 — Hosted Model

The zero-friction path: a litellm model name. `openai/text-embedding-3-small`
produces 1536-dimensional vectors and is the workhorse for retrieval demos. The
`batch_size` controls how many texts are sent per API call when you embed long lists.

In [3]:
# pip install dspy[numpy] — numpy is already in this project
embedder = dspy.Embedder("openai/text-embedding-3-small", batch_size=100)
embeddings = embedder(["hello", "world"])

assert embeddings.shape == (2, 1536)
print(f"shape: {embeddings.shape}  (2 texts x 1536-dim vectors)")
print(f"first 5 dims of 'hello': {embeddings[0][:5]}")

shape: (2, 1536)  (2 texts x 1536-dim vectors)
first 5 dims of 'hello': [ 0.01675415 -0.05575562  0.00563431  0.06622314  0.00894165]


## 4. Official Example 2 — Local Model (sentence-transformers)

Any local model works by passing its `encode` function — nothing else changes.
`static-retrieval-mrl-en-v1` is a tiny retrieval model that runs on CPU, fully offline:
no API key, no network, no per-call cost. This is the pattern for private documents or
air-gapped deployments. (The `sentence-transformers` package was added to this project for
this example.)

In [4]:
from sentence_transformers import SentenceTransformer

# Load an extremely efficient local model for retrieval
model = SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cpu")

local_embedder = dspy.Embedder(model.encode)
embeddings = local_embedder(["hello", "world"], batch_size=1)

assert embeddings.shape == (2, 1024)
print(f"shape: {embeddings.shape}  (2 texts x 1024-dim vectors)")
print("computed locally — no API key, no network, no cost")

modules.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/670k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

0_StaticEmbedding/model.safetensors: reconstructing file:   0%|          |  0.00B /  125MB            

0_StaticEmbedding/model.safetensors: downloading bytes:           |  0.00B            

shape: (2, 1024)  (2 texts x 1024-dim vectors)
computed locally — no API key, no network, no cost


## 5. Official Example 3 — Custom Function

The escape hatch: any callable matching the contract works. Here a toy function
returns random 10-dim vectors — but in production this is exactly how you plug in Azure
OpenAI embeddings, Cohere, Voyage, or an in-house embedding service behind your own
auth, retries, and routing. `dspy.Embedder` doesn't care what happens inside, only that
it gets rows of floats back.

In [5]:
def my_embedder(texts):
    return np.random.rand(len(texts), 10)


custom_embedder = dspy.Embedder(my_embedder)
embeddings = custom_embedder(["hello", "world"], batch_size=1)

assert embeddings.shape == (2, 10)
print(f"shape: {embeddings.shape}")
print("plug in ANY embedding backend (Azure OpenAI, Cohere, your own service) this way")

shape: (2, 10)
plug in ANY embedding backend (Azure OpenAI, Cohere, your own service) this way


## 6. Payoff — Seeing Semantics in the Vectors

Cosine similarity (dot product of normalized vectors) measures the angle between two
embedding vectors: 1.0 = identical direction, 0 = unrelated, −1 = opposite. Three tiny
hosted calls make the property visible.

In [6]:
phrases = [
    "the cat sat on the mat",
    "a kitten rested on the rug",
    "quarterly revenue grew by twelve percent",
]

vectors = dspy.Embedder("openai/text-embedding-3-small")(phrases)
norms = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
similarity = norms @ norms.T

labels = ["cat/mat", "kitten/rug", "revenue"]
print(f"{'':>12}" + "".join(f"{label:>14}" for label in labels))
for label, row in zip(labels, similarity, strict=True):
    print(f"{label:>12}" + "".join(f"{value:>14.2f}" for value in row))

print()
print("The two cat sentences score high with each other and low with finance —")
print("the property retrievers (and the RAG pipeline in Notebook 3) exploit.")

                   cat/mat    kitten/rug       revenue
     cat/mat          1.00          0.60          0.10
  kitten/rug          0.60          1.00          0.10
     revenue          0.10          0.10          1.00

The two cat sentences score high with each other and low with finance —
the property retrievers (and the RAG pipeline in Notebook 3) exploit.


## Where Embeddings Show Up in DSPy

- **Retrieval / RAG** — retriever modules embed queries and passages to find relevant
  context; see Notebook 3 (RAG with Assertions) for the full pipeline.
- **Caching** — `caching=True` (default) means re-embedding the same texts is free; only new
  texts cost API calls.
- **Async pipelines** — use `await embedder.acall(texts)` to embed concurrently in
  production services.
- **Switching providers** — change `EMBEDDING_MODEL` in `.env` (e.g.
  `gemini/text-embedding-004`); litellm routes it, no code changes.

**End of the DSPy module catalog series — Notebooks 1, 2, and 3 next for signatures,
optimizers, and RAG.**